# Patient-level composition and program-score reanalysis

**Purpose:** replace region-level condition inference with source-backed patient-level analysis for:

- final CNV-informed compartments;
- malignant-state composition;
- overall malignant program scores.

The independent unit is the source-backed patient. Multiregional fragments are combined within patient before statistical testing. No patient is treated as paired across conditions because GSE182109 contains distinct LGG, newly diagnosed GBM, and recurrent GBM patients.

This notebook is a sensitivity and decision notebook. It does not overwrite canonical objects or historical result tables. It writes versioned outputs to:

`metadata/sensitivity/patient_level_composition_program_reanalysis/`

Statistical interpretation is deliberately conservative:

- all four-condition inference remains exploratory because LGG has 2 patients and rGBM has 5 patients;
- standard Kruskal–Wallis results are accompanied by label-permutation P values;
- pairwise results include Mann–Whitney P values, rank-biserial effect sizes, and bootstrap confidence intervals;
- manuscript decisions distinguish retained, weakened, unsupported, and historically nonsignificant findings;
- no cell-level P value is used as patient-level evidence.


In [1]:
from pathlib import Path
import hashlib
import json
import os
import sys
from datetime import datetime
from itertools import combinations

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 300)

SEED = 1337
N_GLOBAL_PERMUTATIONS = 20000
N_BOOTSTRAP = 5000
CONDITION_ORDER = ["GBM", "LGG", "ndGBM", "rGBM"]

NOTEBOOK_NAME = "03_patient_level_composition_program_reanalysis.ipynb"
DESIGN_REL = Path("metadata/sensitivity/source_backed_study_design/source_backed_sample_design.tsv")
OUT_REL = Path("metadata/sensitivity/patient_level_composition_program_reanalysis")

INPUTS = {
    "compartment": Path("metadata/DA_sample_level_compartment_proportions_HVG8000_r0_6.tsv"),
    "state": Path("metadata/malignant_state_sample_level_proportions_HVG8000_r0_6.tsv"),
    "program": Path("metadata/condition_malignant_pseudobulk_DE_HVG8000_r0_6_sample_level_program_scores.tsv"),
    "historical_compartment_global": Path("metadata/DA_global_kruskal_compartment_sample_level_HVG8000_r0_6.tsv"),
    "historical_state_global": Path("metadata/malignant_state_global_kruskal_sample_level_HVG8000_r0_6.tsv"),
    "historical_program_global": Path("metadata/condition_malignant_pseudobulk_DE_HVG8000_r0_6_program_score_global_kruskal.tsv"),
}

def resolve_root():
    candidates=[]
    if os.environ.get("GLIOMA_PROJECT_ROOT"):
        candidates.append(Path(os.environ["GLIOMA_PROJECT_ROOT"]).expanduser())
    here=Path.cwd().resolve()
    candidates.extend([here,*here.parents,Path(r"G:\Repository\glioma-cnv-single-cell-atlas")])
    for p in candidates:
        if (p/DESIGN_REL).exists() and all((p/x).exists() for x in INPUTS.values()):
            return p.resolve()
    raise FileNotFoundError("Could not resolve repository root or required inputs")

ROOT=resolve_root()
DESIGN=ROOT/DESIGN_REL
OUT_DIR=ROOT/OUT_REL
OUT_DIR.mkdir(parents=True,exist_ok=True)
PATHS={k:ROOT/v for k,v in INPUTS.items()}

print("Repository root:",ROOT)
print("Output directory:",OUT_DIR)


Repository root: G:\Repository\glioma-cnv-single-cell-atlas
Output directory: G:\Repository\glioma-cnv-single-cell-atlas\metadata\sensitivity\patient_level_composition_program_reanalysis


## 1 Load the source-backed design and verify independent units


In [2]:
design=pd.read_csv(DESIGN,sep="\t")
required={"sample_id","core_dataset","condition","n_cells","source_backed_patient_id","source_patient_label"}
missing=required-set(design.columns)
if missing:
    raise AssertionError(f"Design manifest missing columns: {sorted(missing)}")

checks={
    "sample_rows":len(design),
    "unique_samples":design["sample_id"].nunique(),
    "unique_patients":design["source_backed_patient_id"].nunique(),
    "patients_spanning_conditions":int((design.groupby("source_backed_patient_id")["condition"].nunique()>1).sum()),
}
expected={"sample_rows":83,"unique_samples":83,"unique_patients":57,"patients_spanning_conditions":0}
if checks!=expected:
    raise AssertionError(f"Source-backed design checkpoint failed: {checks}")

patient_counts=(design.groupby("condition")["source_backed_patient_id"].nunique().reindex(CONDITION_ORDER))
if patient_counts.to_dict()!={"GBM":39,"LGG":2,"ndGBM":11,"rGBM":5}:
    raise AssertionError(f"Unexpected patient counts: {patient_counts.to_dict()}")

display(patient_counts.rename("n_patients").reset_index())


,condition,n_patients
0,GBM,39
1,LGG,2
2,ndGBM,11
3,rGBM,5


## 2 Statistical helper functions

The global label-permutation test shuffles condition labels across independent patients while preserving group sizes. Pairwise rank-biserial correlation is reported on a scale from −1 to 1. Positive values mean larger values in the first named condition. Bootstrap intervals describe effect-size uncertainty and are not used to rescue significance.


In [3]:
def bh(values):
    values=np.asarray(values,dtype=float)
    out=np.full(values.shape,np.nan)
    mask=np.isfinite(values)
    if mask.any():
        out[mask]=multipletests(values[mask],method="fdr_bh")[1]
    return out

def rank_biserial(x,y):
    x=np.asarray(x,float); y=np.asarray(y,float)
    u=stats.mannwhitneyu(x,y,alternative="two-sided",method="auto").statistic
    return float(2*u/(len(x)*len(y))-1)

def bootstrap_rbc_ci(x,y,seed,n_boot=N_BOOTSTRAP):
    x=np.asarray(x,float); y=np.asarray(y,float)
    rng=np.random.default_rng(seed)
    vals=np.empty(n_boot,dtype=float)
    for i in range(n_boot):
        xb=rng.choice(x,size=len(x),replace=True)
        yb=rng.choice(y,size=len(y),replace=True)
        vals[i]=rank_biserial(xb,yb)
    return np.quantile(vals,[0.025,0.975])

def permuted_kruskal(values,labels,seed,n_perm=N_GLOBAL_PERMUTATIONS):
    values=np.asarray(values,float); labels=np.asarray(labels,str)
    levels=[x for x in CONDITION_ORDER if x in set(labels)]
    groups=[values[labels==x] for x in levels]
    observed=float(stats.kruskal(*groups).statistic)
    rng=np.random.default_rng(seed)
    exceed=0
    for _ in range(n_perm):
        shuffled=rng.permutation(labels)
        h=float(stats.kruskal(*[values[shuffled==x] for x in levels]).statistic)
        exceed += h >= observed-1e-12
    return observed,(exceed+1)/(n_perm+1)

def global_tests(long_df,feature_col,value_col,family):
    rows=[]
    for idx,(feature,g) in enumerate(long_df.groupby(feature_col,sort=True)):
        arrays=[g.loc[g.condition.eq(c),value_col].dropna().to_numpy(float) for c in CONDITION_ORDER]
        if any(len(x)==0 for x in arrays):
            continue
        h,p=stats.kruskal(*arrays)
        hp,pp=permuted_kruskal(g[value_col],g["condition"],SEED+idx)
        row={"analysis_family":family,"feature":feature,"value":value_col,
             "n_total_patients":len(g),"kruskal_H":h,"kruskal_p":p,
             "permutation_H":hp,"permutation_p":pp}
        for c,a in zip(CONDITION_ORDER,arrays):
            row[f"n_{c}"]=len(a); row[f"median_{c}"]=float(np.median(a)); row[f"iqr_{c}"]=float(np.subtract(*np.percentile(a,[75,25])))
        rows.append(row)
    out=pd.DataFrame(rows)
    out["kruskal_q_bh"]=bh(out["kruskal_p"])
    out["permutation_q_bh"]=bh(out["permutation_p"])
    return out

def pairwise_tests(long_df,feature_col,value_col,family):
    rows=[]; counter=0
    for feature,g in long_df.groupby(feature_col,sort=True):
        for c1,c2 in combinations(CONDITION_ORDER,2):
            x=g.loc[g.condition.eq(c1),value_col].dropna().to_numpy(float)
            y=g.loc[g.condition.eq(c2),value_col].dropna().to_numpy(float)
            if len(x)==0 or len(y)==0:
                continue
            test=stats.mannwhitneyu(x,y,alternative="two-sided",method="auto")
            effect=rank_biserial(x,y)
            lo,hi=bootstrap_rbc_ci(x,y,SEED+counter)
            counter+=1
            rows.append({"analysis_family":family,"feature":feature,"condition_1":c1,"condition_2":c2,
                         "n_1":len(x),"n_2":len(y),"median_1":np.median(x),"median_2":np.median(y),
                         "median_difference_1_minus_2":np.median(x)-np.median(y),
                         "mannwhitney_U":test.statistic,"p_value":test.pvalue,
                         "rank_biserial_1_minus_2":effect,"rbc_ci_low":lo,"rbc_ci_high":hi})
    out=pd.DataFrame(rows)
    out["q_value_bh_family"]=bh(out["p_value"])
    return out


## 3 Patient-level CNV-informed compartment composition

All four compartment categories are completed for every sample, including explicit zeros. Numerators and denominators are then summed across regional fragments from the same source-backed patient.


In [4]:
comp=pd.read_csv(PATHS["compartment"],sep="\t")
sample_meta=comp[["unit_id","condition","dataset","patient","n_cells_total"]].drop_duplicates("unit_id")
features=sorted(comp["feature"].unique())
grid=(sample_meta.assign(_key=1).merge(pd.DataFrame({"feature":features,"_key":1}),on="_key").drop(columns="_key"))
comp_complete=grid.merge(comp[["unit_id","feature","n_cells_feature"]],on=["unit_id","feature"],how="left")
comp_complete["n_cells_feature"]=comp_complete["n_cells_feature"].fillna(0).astype(np.int64)
comp_complete=comp_complete.merge(design[["sample_id","source_backed_patient_id","source_patient_label"]],left_on="unit_id",right_on="sample_id",how="left",validate="many_to_one")
if comp_complete["source_backed_patient_id"].isna().any(): raise AssertionError("Unmapped compartment samples")

patient_comp=(comp_complete.groupby(["source_backed_patient_id","source_patient_label","condition","dataset","feature"],as_index=False)
              .agg(n_cells_feature=("n_cells_feature","sum"),n_cells_total=("n_cells_total","sum"),n_regional_samples=("sample_id","nunique")))
patient_comp["proportion"]=patient_comp["n_cells_feature"]/patient_comp["n_cells_total"]
if patient_comp["source_backed_patient_id"].nunique()!=57: raise AssertionError("Compartment patient count failed")
if not np.allclose(patient_comp.groupby("source_backed_patient_id")["proportion"].sum().to_numpy(),1.0): raise AssertionError("Compartment proportions do not sum to 1")

comp_global=global_tests(patient_comp,"feature","proportion","compartment")
comp_pairwise=pairwise_tests(patient_comp,"feature","proportion","compartment")
display(comp_global)


,analysis_family,feature,value,n_total_patients,kruskal_H,kruskal_p,permutation_H,permutation_p,n_GBM,median_GBM,iqr_GBM,n_LGG,median_LGG,iqr_LGG,n_ndGBM,median_ndGBM,iqr_ndGBM,n_rGBM,median_rGBM,iqr_rGBM,kruskal_q_bh,permutation_q_bh
0,compartment,cycling_malignant_like_candidate,proportion,57,3.410268,0.332588,3.410268,0.348183,39,0.107757,0.142368,2,0.021021,0.002874,11,0.049381,0.129640,5,0.111224,0.117348,0.443451,0.464243
1,compartment,malignant_like,proportion,57,8.391073,0.038584,8.391073,0.026749,39,0.527081,0.485107,2,0.353076,0.122577,11,0.249805,0.203638,5,0.321547,0.318079,0.077169,0.053497
2,compartment,non_malignant_or_diploid_leaning,proportion,57,1.538641,0.673381,1.538641,0.702515,39,0.037133,0.064247,2,0.102297,0.052101,11,0.039530,0.069822,5,0.037075,0.029144,0.673381,0.702515
3,compartment,review,proportion,57,11.175690,0.010813,11.175690,0.004250,39,0.205074,0.494537,2,0.523606,0.073350,11,0.601831,0.326660,5,0.535691,0.495064,0.043251,0.016999


## 4 Patient-level malignant-state composition

State counts and the malignant-state-pool denominator are summed across regional fragments. Fractions are calculated only after aggregation.


In [5]:
state=pd.read_csv(PATHS["state"],sep="\t")
state=state.merge(design[["sample_id","source_backed_patient_id","source_patient_label"]],on="sample_id",how="left",validate="many_to_one")
if state["source_backed_patient_id"].isna().any(): raise AssertionError("Unmapped state samples")

patient_state=(state.groupby(["source_backed_patient_id","source_patient_label","condition","core_dataset","malignant_state"],as_index=False)
               .agg(n_state_cells=("n_state_cells","sum"),
                    n_malignant_state_analysis_cells=("n_malignant_state_analysis_cells","sum"),
                    n_total_cells=("n_total_cells_in_sample","sum"),
                    n_regional_samples=("sample_id","nunique")))
patient_state["fraction_within_malignant_state_pool"]=patient_state["n_state_cells"]/patient_state["n_malignant_state_analysis_cells"]
patient_state["fraction_of_all_cells"]=patient_state["n_state_cells"]/patient_state["n_total_cells"]
if patient_state["source_backed_patient_id"].nunique()!=57:
    raise AssertionError(f"Expected 57 patients with malignant-state cells, found {patient_state['source_backed_patient_id'].nunique()}")
if not np.allclose(patient_state.groupby("source_backed_patient_id")["fraction_within_malignant_state_pool"].sum().to_numpy(),1.0): raise AssertionError("State fractions do not sum to 1")

state_global=global_tests(patient_state,"malignant_state","fraction_within_malignant_state_pool","malignant_state")
state_pairwise=pairwise_tests(patient_state,"malignant_state","fraction_within_malignant_state_pool","malignant_state")
display(state_global)


,analysis_family,feature,value,n_total_patients,kruskal_H,kruskal_p,permutation_H,permutation_p,n_GBM,median_GBM,iqr_GBM,n_LGG,median_LGG,iqr_LGG,n_ndGBM,median_ndGBM,iqr_ndGBM,n_rGBM,median_rGBM,iqr_rGBM,kruskal_q_bh,permutation_q_bh
0,malignant_state,AC_like_reactive_malignant,fraction_within_malignant_state_pool,57,2.768317,0.428742,2.768317,0.451677,39,0.093506,0.229581,2,0.139566,0.119410,11,0.171748,0.337924,5,0.107549,0.026869,0.428742,0.451677
1,malignant_state,NPC_neuronal_like_malignant,fraction_within_malignant_state_pool,57,3.131141,0.371844,3.131141,0.391780,39,0.017964,0.057396,2,0.033380,0.032203,11,0.004677,0.014071,5,0.006086,0.005194,0.428742,0.451677
2,malignant_state,OPC_NPC_like_malignant,fraction_within_malignant_state_pool,57,5.827789,0.120296,5.827789,0.110844,39,0.299812,0.232966,2,0.756756,0.106962,11,0.388462,0.119939,5,0.413793,0.254597,0.318935,0.295735
3,malignant_state,cycling_malignant_candidate,fraction_within_malignant_state_pool,57,5.692385,0.127574,5.692385,0.118294,39,0.203187,0.124946,2,0.060409,0.012575,11,0.180699,0.189278,5,0.289435,0.132160,0.318935,0.295735
4,malignant_state,hypoxia_stress_malignant,fraction_within_malignant_state_pool,57,4.453047,0.216517,4.453047,0.224539,39,0.127023,0.327353,2,0.009888,0.007181,11,0.152850,0.223326,5,0.223162,0.286730,0.360861,0.374231


## 5 Patient-level overall malignant program scores

Sample-level overall malignant program scores are combined across regional fragments using the number of malignant cells as weights. These are overall malignant scores, not state-specific scores.


In [6]:
program=pd.read_csv(PATHS["program"],sep="\t")
program=program.merge(design[["sample_id","core_dataset","source_backed_patient_id","source_patient_label"]],on="sample_id",how="left",validate="many_to_one")
if program["source_backed_patient_id"].isna().any(): raise AssertionError("Unmapped program-score samples")
score_cols=[c for c in program.columns if c.endswith("_score")]

rows=[]
for keys,g in program.groupby(["source_backed_patient_id","source_patient_label","condition","core_dataset"],sort=True):
    w=g["n_cells"].to_numpy(float)
    row=dict(zip(["source_backed_patient_id","source_patient_label","condition","core_dataset"],keys))
    row["n_cells"]=int(w.sum()); row["n_regional_samples"]=g["sample_id"].nunique()
    for col in score_cols: row[col]=float(np.average(g[col].to_numpy(float),weights=w))
    rows.append(row)
patient_program_wide=pd.DataFrame(rows)
patient_program=patient_program_wide.melt(id_vars=["source_backed_patient_id","source_patient_label","condition","core_dataset","n_cells","n_regional_samples"],value_vars=score_cols,var_name="program",value_name="score")
if patient_program["source_backed_patient_id"].nunique()!=56:
    raise AssertionError(f"Expected 56 patients with qualifying overall malignant pseudobulks, found {patient_program['source_backed_patient_id'].nunique()}")

program_global=global_tests(patient_program,"program","score","program_score")
program_pairwise=pairwise_tests(patient_program,"program","score","program_score")
display(program_global)


,analysis_family,feature,value,n_total_patients,kruskal_H,kruskal_p,permutation_H,permutation_p,n_GBM,median_GBM,iqr_GBM,n_LGG,median_LGG,iqr_LGG,n_ndGBM,median_ndGBM,iqr_ndGBM,n_rGBM,median_rGBM,iqr_rGBM,kruskal_q_bh,permutation_q_bh
0,program_score,AC_reactive_score,score,56,1.998535,0.572711,1.998535,0.603670,39,-0.042457,0.909457,2,-0.576837,0.653442,10,0.263926,0.639180,5,0.103808,0.287433,0.594439,0.621969
1,program_score,MES_like_score,score,56,6.182745,0.103050,6.182745,0.090395,39,0.080356,0.867712,2,-2.039097,0.344610,10,-0.110176,1.307798,5,-0.031560,0.603554,0.303083,0.282436
2,program_score,NPC_neural_score,score,56,1.895209,0.594439,1.895209,0.621969,39,-0.099047,1.177361,2,0.541745,0.365454,10,-0.279936,1.042308,5,-0.155421,0.768205,0.594439,0.621969
3,program_score,OPC_proneural_score,score,56,4.013659,0.259993,4.013659,0.271236,39,0.022480,0.687947,2,0.864401,0.309206,10,0.185916,0.626775,5,0.309450,1.436824,0.363990,0.379731
4,program_score,cycling_score,score,56,4.253104,0.235399,4.253104,0.242438,39,0.203176,0.679323,2,-0.824695,0.576323,10,0.416079,0.525736,5,0.412889,0.131161,0.363990,0.379731
5,program_score,hypoxia_stress_score,score,56,5.650790,0.129893,5.650790,0.121044,39,0.139258,1.321252,2,-1.337867,0.089332,10,-0.004100,1.102088,5,-0.000114,1.213100,0.303083,0.282436
6,program_score,immune_myeloid_review_score,score,56,6.017794,0.110748,6.017794,0.099695,39,0.383883,1.587315,2,-0.956951,0.189357,10,-0.296425,0.688247,5,-0.597110,0.756024,0.303083,0.282436


## 6 Reproduce the previous patient-condition sensitivity tables

The previous notebook used dataset, numeric patient identifier, and condition. Because condition separated the five false numeric collisions, its aggregated numerical values should agree with the new source-backed patient aggregation. This section verifies that equivalence.


In [7]:
PRIOR=ROOT/"metadata/sensitivity/patient_condition_sensitivity"
audit=[]
legacy=design[["condition","patient_id","source_backed_patient_id"]].drop_duplicates()

# Compartment: the new table deliberately completes three historically absent zero rows.
old=pd.read_csv(PRIOR/"patient_condition_compartment_proportions.tsv",sep="\t")
n=patient_comp.merge(legacy,on=["source_backed_patient_id","condition"],how="left",validate="many_to_one")
n=n[["dataset","patient_id","condition","feature","n_cells_feature","n_cells_total","proportion"]].rename(columns={"patient_id":"patient","n_cells_feature":"new_feature_cells","n_cells_total":"new_total_cells","proportion":"new_value"})
o=old[["dataset","patient","condition","feature","n_cells_feature","n_cells_total","proportion"]].rename(columns={"n_cells_feature":"old_feature_cells","n_cells_total":"old_total_cells","proportion":"old_value"})
j=n.merge(o,on=["dataset","patient","condition","feature"],how="outer",indicator=True)
shared=j["_merge"].eq("both")
extra_new=j["_merge"].eq("left_only")
maxdiff=float(np.max(np.abs(j.loc[shared,"new_value"]-j.loc[shared,"old_value"])))
extra_zero=bool((j.loc[extra_new,"new_feature_cells"]==0).all() and (j.loc[extra_new,"new_value"]==0).all())
numerators_match=bool((j.loc[shared,"new_feature_cells"]==j.loc[shared,"old_feature_cells"]).all())
denominators_not_smaller=bool((j.loc[shared,"new_total_cells"]>=j.loc[shared,"old_total_cells"]).all())
same_denominator=shared & j["new_total_cells"].eq(j["old_total_cells"])
same_denominator_values_match=bool(np.allclose(j.loc[same_denominator,"new_value"],j.loc[same_denominator,"old_value"],rtol=0,atol=1e-12))
denominator_corrected_rows=int((shared & j["new_total_cells"].gt(j["old_total_cells"])).sum())
audit.append({"prior_file":"patient_condition_compartment_proportions.tsv","new_rows":len(n),"old_rows":len(o),"joined_rows":len(j),"shared_rows":int(shared.sum()),"new_explicit_zero_rows":int(extra_new.sum()),"denominator_corrected_rows":denominator_corrected_rows,"max_abs_difference":maxdiff,"pass":bool((~j["_merge"].eq("right_only")).all() and extra_zero and numerators_match and denominators_not_smaller and same_denominator_values_match)})

# Malignant states: prior and new tables should match exactly after schema normalization.
old=pd.read_csv(PRIOR/"patient_condition_malignant_state_proportions.tsv",sep="\t")
n=patient_state.merge(legacy,on=["source_backed_patient_id","condition"],how="left",validate="many_to_one")
n=n[["core_dataset","patient_id","condition","malignant_state","fraction_within_malignant_state_pool"]].rename(columns={"core_dataset":"dataset","patient_id":"patient","malignant_state":"feature","fraction_within_malignant_state_pool":"new_value"})
o=old[["core_dataset","patient_id","condition","malignant_state","fraction_within_malignant_state_pool"]].rename(columns={"core_dataset":"dataset","patient_id":"patient","malignant_state":"feature","fraction_within_malignant_state_pool":"old_value"})
j=n.merge(o,on=["dataset","patient","condition","feature"],how="outer",indicator=True)
shared=j["_merge"].eq("both"); maxdiff=float(np.max(np.abs(j.loc[shared,"new_value"]-j.loc[shared,"old_value"])))
audit.append({"prior_file":"patient_condition_malignant_state_proportions.tsv","new_rows":len(n),"old_rows":len(o),"joined_rows":len(j),"shared_rows":int(shared.sum()),"new_explicit_zero_rows":0,"denominator_corrected_rows":0,"max_abs_difference":maxdiff,"pass":bool((j["_merge"]=="both").all() and maxdiff<1e-12)})

# Programs: the prior table is wide, so reshape it before comparison.
old=pd.read_csv(PRIOR/"patient_condition_program_scores.tsv",sep="\t")
o=old.melt(id_vars=["dataset","patient","condition"],value_vars=score_cols,var_name="feature",value_name="old_value")
n=patient_program.merge(legacy,on=["source_backed_patient_id","condition"],how="left",validate="many_to_one")
n=n[["core_dataset","patient_id","condition","program","score"]].rename(columns={"core_dataset":"dataset","patient_id":"patient","program":"feature","score":"new_value"})
j=n.merge(o,on=["dataset","patient","condition","feature"],how="outer",indicator=True)
shared=j["_merge"].eq("both"); maxdiff=float(np.max(np.abs(j.loc[shared,"new_value"]-j.loc[shared,"old_value"])))
audit.append({"prior_file":"patient_condition_program_scores.tsv","new_rows":len(n),"old_rows":len(o),"joined_rows":len(j),"shared_rows":int(shared.sum()),"new_explicit_zero_rows":0,"denominator_corrected_rows":0,"max_abs_difference":maxdiff,"pass":bool((j["_merge"]=="both").all() and maxdiff<1e-12)})

prior_reproduction=pd.DataFrame(audit)
display(prior_reproduction)
if not prior_reproduction["pass"].all(): raise AssertionError("Previous sensitivity reproduction failed")


,prior_file,new_rows,old_rows,joined_rows,shared_rows,new_explicit_zero_rows,denominator_corrected_rows,max_abs_difference,pass
0,patient_condition_compartment_proportions.tsv,228,225,228,225,3,5,5.136993e-02,True
1,patient_condition_malignant_state_proportions.tsv,285,285,285,285,0,0,1.110223e-16,True
2,patient_condition_program_scores.tsv,392,392,392,392,0,0,4.440892e-16,True


## 7 Manuscript decision matrix

Historical sample-level q values are compared with patient-level permutation q values. A result can remain biologically descriptive even when its inferential status is not retained.


In [8]:
hist_comp=pd.read_csv(PATHS["historical_compartment_global"],sep="\t")
hist_state=pd.read_csv(PATHS["historical_state_global"],sep="\t")
hist_prog=pd.read_csv(PATHS["historical_program_global"],sep="\t")

def find_col(df,candidates):
    for c in candidates:
        if c in df.columns:return c
    raise KeyError(candidates)

hc_feature=find_col(hist_comp,["feature","final_compartment_major"]); hc_q=find_col(hist_comp,["qvalue_fdr_bh","q_value_bh","padj","q_value"])
hs_feature="malignant_state"; hs_q="q_value_bh"
hp_feature="score_col"; hp_q="padj"

historical=pd.concat([
    hist_comp[[hc_feature,hc_q]].rename(columns={hc_feature:"feature",hc_q:"historical_q"}).assign(analysis_family="compartment"),
    hist_state[[hs_feature,hs_q]].rename(columns={hs_feature:"feature",hs_q:"historical_q"}).assign(analysis_family="malignant_state"),
    hist_prog[[hp_feature,hp_q]].rename(columns={hp_feature:"feature",hp_q:"historical_q"}).assign(analysis_family="program_score"),
],ignore_index=True)

patient_global=pd.concat([
    comp_global[["analysis_family","feature","kruskal_p","kruskal_q_bh","permutation_p","permutation_q_bh"]],
    state_global[["analysis_family","feature","kruskal_p","kruskal_q_bh","permutation_p","permutation_q_bh"]],
    program_global[["analysis_family","feature","kruskal_p","kruskal_q_bh","permutation_p","permutation_q_bh"]],
],ignore_index=True)

decision=historical.merge(patient_global,on=["analysis_family","feature"],how="outer",indicator=True)
if not (decision["_merge"]=="both").all():
    display(decision[decision["_merge"]!="both"])
    raise AssertionError("Historical/patient feature mapping failed")

def classify(row):
    old=row.historical_q<0.05; new=row.permutation_q_bh<0.05
    if old and new:return "retained_after_patient_aggregation"
    if old and not new:return "unsupported_after_patient_aggregation"
    if not old and new:return "new_patient_level_signal_requires_review"
    return "remains_nonsignificant"

decision["decision_status"]=decision.apply(classify,axis=1)
decision["claim_strength"]="exploratory because of small LGG/rGBM groups and condition-dataset confounding"
decision=decision.drop(columns="_merge").sort_values(["analysis_family","feature"]).reset_index(drop=True)
display(decision)
display(decision.groupby(["analysis_family","decision_status"]).size().rename("n_features").reset_index())


,feature,historical_q,analysis_family,kruskal_p,kruskal_q_bh,permutation_p,permutation_q_bh,decision_status,claim_strength
0,cycling_malignant_like_candidate,0.313933,compartment,0.332588,0.443451,0.348183,0.464243,remains_nonsignificant,exploratory because of small LGG/rGBM groups a...
1,malignant_like,0.011215,compartment,0.038584,0.077169,0.026749,0.053497,unsupported_after_patient_aggregation,exploratory because of small LGG/rGBM groups a...
2,non_malignant_or_diploid_leaning,0.632788,compartment,0.673381,0.673381,0.702515,0.702515,remains_nonsignificant,exploratory because of small LGG/rGBM groups a...
3,review,0.001145,compartment,0.010813,0.043251,0.004250,0.016999,retained_after_patient_aggregation,exploratory because of small LGG/rGBM groups a...
4,AC_like_reactive_malignant,0.082393,malignant_state,0.428742,0.428742,0.451677,0.451677,remains_nonsignificant,exploratory because of small LGG/rGBM groups a...
5,NPC_neuronal_like_malignant,0.068123,malignant_state,0.371844,0.428742,0.391780,0.451677,remains_nonsignificant,exploratory because of small LGG/rGBM groups a...
6,OPC_NPC_like_malignant,0.020816,malignant_state,0.120296,0.318935,0.110844,0.295735,unsupported_after_patient_aggregation,exploratory because of small LGG/rGBM groups a...
7,cycling_malignant_candidate,0.020816,malignant_state,0.127574,0.318935,0.118294,0.295735,unsupported_after_patient_aggregation,exploratory because of small LGG/rGBM groups a...
8,hypoxia_stress_malignant,0.025564,malignant_state,0.216517,0.360861,0.224539,0.374231,unsupported_after_patient_aggregation,exploratory because of small LGG/rGBM groups a...
9,AC_reactive_score,0.098234,program_score,0.572711,0.594439,0.603670,0.621969,remains_nonsignificant,exploratory because of small LGG/rGBM groups a...


,analysis_family,decision_status,n_features
0,compartment,remains_nonsignificant,2
1,compartment,retained_after_patient_aggregation,1
2,compartment,unsupported_after_patient_aggregation,1
3,malignant_state,remains_nonsignificant,2
4,malignant_state,unsupported_after_patient_aggregation,3
5,program_score,remains_nonsignificant,1
6,program_score,unsupported_after_patient_aggregation,6


## 8 Write audited outputs


In [9]:
outputs={}
def write_tsv(df,name):
    p=OUT_DIR/name; df.to_csv(p,sep="\t",index=False); outputs[name]=p

for df,name in [
    (patient_comp,"patient_compartment_proportions.tsv"),(comp_global,"patient_compartment_global_tests.tsv"),(comp_pairwise,"patient_compartment_pairwise_tests.tsv"),
    (patient_state,"patient_malignant_state_proportions.tsv"),(state_global,"patient_malignant_state_global_tests.tsv"),(state_pairwise,"patient_malignant_state_pairwise_tests.tsv"),
    (patient_program_wide,"patient_program_scores_wide.tsv"),(patient_program,"patient_program_scores_long.tsv"),(program_global,"patient_program_global_tests.tsv"),(program_pairwise,"patient_program_pairwise_tests.tsv"),
    (prior_reproduction,"prior_sensitivity_reproduction.tsv"),(decision,"manuscript_decision_matrix.tsv"),
]: write_tsv(df,name)

summary=(decision.groupby(["analysis_family","decision_status"],as_index=False).size().rename(columns={"size":"n_features"}))
write_tsv(summary,"decision_summary.tsv")

readiness=pd.DataFrame([
    ("source_backed_patients",57,design["source_backed_patient_id"].nunique()),
    ("patients_spanning_conditions",0,int((design.groupby("source_backed_patient_id")["condition"].nunique()>1).sum())),
    ("compartment_patients",57,patient_comp["source_backed_patient_id"].nunique()),
    ("state_patients",57,patient_state["source_backed_patient_id"].nunique()),
    ("program_patients",56,patient_program["source_backed_patient_id"].nunique()),
    ("prior_sensitivity_all_reproduced",True,bool(prior_reproduction["pass"].all())),
    ("decision_features",16,len(decision)),
    ("canonical_files_modified",False,False),
],columns=["check","expected","observed"])
readiness["pass"]=readiness["expected"]==readiness["observed"]
write_tsv(readiness,"readiness_checks.tsv")

run_info={"notebook":NOTEBOOK_NAME,"timestamp":datetime.now().astimezone().isoformat(),"python":sys.version,
          "repository_root":str(ROOT),"output_directory":str(OUT_DIR.relative_to(ROOT)),"seed":SEED,
          "n_global_permutations":N_GLOBAL_PERMUTATIONS,"n_bootstrap":N_BOOTSTRAP,"canonical_files_modified":False}
(OUT_DIR/"run_info.json").write_text(json.dumps(run_info,indent=2),encoding="utf-8"); outputs["run_info.json"]=OUT_DIR/"run_info.json"

manifest=[]
for name,p in sorted(outputs.items()):
    b=p.read_bytes(); rows=cols=np.nan
    if p.suffix==".tsv": rows,cols=pd.read_csv(p,sep="\t").shape
    manifest.append({"file":name,"relative_path":str(p.relative_to(ROOT)),"rows":rows,"columns":cols,"bytes":len(b),"sha256":hashlib.sha256(b).hexdigest()})
manifest=pd.DataFrame(manifest); manifest.to_csv(OUT_DIR/"output_manifest.tsv",sep="\t",index=False)

display(readiness); display(summary); display(manifest)
if not readiness["pass"].all(): raise AssertionError("Final readiness checks failed")
print("PATIENT-LEVEL COMPOSITION AND PROGRAM REANALYSIS PASSED")
print("Outputs:",OUT_DIR)


,check,expected,observed,pass
0,source_backed_patients,57,57,True
1,patients_spanning_conditions,0,0,True
2,compartment_patients,57,57,True
3,state_patients,57,57,True
4,program_patients,56,56,True
5,prior_sensitivity_all_reproduced,True,True,True
6,decision_features,16,16,True
7,canonical_files_modified,False,False,True


,analysis_family,decision_status,n_features
0,compartment,remains_nonsignificant,2
1,compartment,retained_after_patient_aggregation,1
2,compartment,unsupported_after_patient_aggregation,1
3,malignant_state,remains_nonsignificant,2
4,malignant_state,unsupported_after_patient_aggregation,3
5,program_score,remains_nonsignificant,1
6,program_score,unsupported_after_patient_aggregation,6


,file,relative_path,rows,columns,bytes,sha256
0,decision_summary.tsv,metadata\sensitivity\patient_level_composition...,7.0,3.0,379,096da773dc8ac505e1990c0a61e143ca003de035cdbad6...
1,manuscript_decision_matrix.tsv,metadata\sensitivity\patient_level_composition...,16.0,9.0,4055,706e14796b8131e401cf38b9677d6ae8dd50253830ea4c...
2,patient_compartment_global_tests.tsv,metadata\sensitivity\patient_level_composition...,4.0,22.0,1560,f9d47d755f3e7bbd2f5b941aace95a93c5491a1618bfae...
3,patient_compartment_pairwise_tests.tsv,metadata\sensitivity\patient_level_composition...,24.0,15.0,4879,e4e358b7dd849c597148f1ca9c59bf0403d246bf5ac236...
4,patient_compartment_proportions.tsv,metadata\sensitivity\patient_level_composition...,228.0,9.0,25276,3e43adfa112946694ea6b7390735377cd6168413390f5b...
5,patient_malignant_state_global_tests.tsv,metadata\sensitivity\patient_level_composition...,5.0,22.0,2075,0db68759060a9b69693485fe0267c3322db3cd88c0caf6...
6,patient_malignant_state_pairwise_tests.tsv,metadata\sensitivity\patient_level_composition...,30.0,15.0,6380,6bffe836e3f6bca8880938dd3d47e09552785f0d08309d...
7,patient_malignant_state_proportions.tsv,metadata\sensitivity\patient_level_composition...,285.0,11.0,39416,93540c19f18722219551591b79e326dd646fd335388d76...
8,patient_program_global_tests.tsv,metadata\sensitivity\patient_level_composition...,7.0,22.0,2489,177e1a22c20eb66f3b55655bb32f8173d02fb544ad93c3...
9,patient_program_pairwise_tests.tsv,metadata\sensitivity\patient_level_composition...,42.0,15.0,8295,6ed29b710d84a4f1d0e9e6275a7587dc05b2d34ef04779...


PATIENT-LEVEL COMPOSITION AND PROGRAM REANALYSIS PASSED
Outputs: G:\Repository\glioma-cnv-single-cell-atlas\metadata\sensitivity\patient_level_composition_program_reanalysis
